# Lab 09 - Agents and Tool Use

**Chapter 6, "Agents"** (tools, planning, function calling, agent failure modes and evaluation) | **Environment:** cluster notebook (local Docker JupyterLab + KU VPN) | **Time:** about 3-3.5 hours (this is a Part-A-only week: side project 2 is due)

You are an AI engineer at Larkspur Ridge IT. Service desk staff keep asking the assistant questions that live in **Harbor PSA data** (Larkspur's ticketing, time, and billing system), not in documents: "Which open tickets does Kaw Valley Auto Group have?", "Who is on TCK-119956?", "What's our response target for that?". Retrieval (Lab 8) can't answer these well. In this lab you give the model **tools** over the ticket database and write the **agent loop** that lets it plan, call tools, read results, and answer.

What you'll build:
1. Five read-only tools over the Larkspur SQLite database, plus their JSON schemas.
2. Native tool calling with the OpenAI `tools` API on `qwen-moe` (the course gateway speaks the same API as OpenAI, so the standard way of offering tools to a model works here).
3. Argument validation that sends useful error messages back to the model.
4. The agent loop (call -> execute -> append result -> repeat, with a step limit).
5. A "write" tool that needs human confirmation (simulated: nothing is ever written).
6. An evaluation on 15 tasks with answers computed from the database (the scorer is provided), step and token counts, failure analysis, and a 3-model comparison.

**Part B (optional bonus):** Co. app v3 - add `tools.py` to your Co. app and route between retrieval and tools.

**Before you start:** KU VPN on, `docker compose up` running, this notebook opened from `labs/lab09-agents/`. Use **Run > Run All Cells** once to see where you stand; every TODO has a placeholder so the notebook runs to the end.

## Words you'll meet in this lab

You don't need to memorize these. Each one is explained again where it first matters; come back here if a word stops you.

| Term | What it means | Why it matters |
|---|---|---|
| **Agent** | A program where the model decides, step by step, which actions to take to reach a goal: it calls a tool, reads the result, and decides what to do next. | It can answer questions that need live data, not just documents. |
| **Tool** | A function the agent is allowed to call. Here, each tool is a small database lookup. | A model can't look anything up on its own; tools give it eyes and hands. |
| **Tool calling (function calling)** | Instead of answering in text, the model replies with the name of a tool and the arguments to pass to it. Your code runs the tool and sends the result back. | This is how an agent acts. The model only *asks*; your code decides what actually runs. |
| **Tool inventory** | The list of tools (names, descriptions, allowed arguments) sent to the model with every request. | The descriptions are the only documentation the model gets, so they shape what it does. |
| **JSON, JSON schema** | JSON is a plain-text format for structured data, like `{"ticket_id": "TCK-119947"}`. A JSON schema describes what valid JSON must look like: which fields, what types, which values are allowed. | Tool arguments arrive as JSON, and you check them against the schema before running anything. |
| **Argument** | A value passed into a function, like the ticket ID passed to `lookup_ticket`. | The model chooses the arguments, so they can be wrong or made up. |
| **Agent loop, step** | The repeating cycle: ask the model, run the tools it asked for, add the results to the conversation, ask again. Each model call is one **step**. | More steps means more time and more tokens; a step limit keeps a confused agent from running forever. |
| **SQL, SQLite, query** | SQL is the language for asking questions of database tables. SQLite is a small database stored in a single file (`larkspur.db`). A query is one SQL request. | Every tool in this lab is one or two SQL queries. |
| **Harbor PSA** | Larkspur's ticketing, time, and billing system (PSA stands for professional services automation). `larkspur.db` is an extract of its data. | The questions in this lab are about this data, not about documents. |
| **Read vs write action** | A read action only looks at data. A write action changes something (closes a ticket, sends an email). | Write actions are where an agent can do real damage. |
| **Human in the loop** | A person must approve an action before it happens. | The standard safety net for write actions (A5). |
| **Trace** | A record of everything the agent did for one question: each tool call, each result, tokens, and the final answer. | You evaluate and debug agents by reading traces. |
| **Prompt and completion tokens** | Prompt tokens: everything sent to the model (input). Completion tokens: what the model wrote back (output). | An agent re-sends the whole conversation every step, so prompt tokens pile up fast. |
| **Router** | Code that decides which path a question takes (here: documents, database, or both). Not the same as the expert router inside a mixture-of-experts model from Lab 02. | Part B's Co. app uses one. |
| **Regular expression (regex)** | A pattern for matching text, such as "TCK- followed by 6 digits". | Used to spot ticket IDs and to check argument formats. |

## Setup

`PACE_S` adds a short pause after each model call. The agent makes several calls per question, so this keeps you well under the 60 requests/minute limit. The whole lab uses about 150 requests.

In [ ]:
import json, os, re, sqlite3, time
from pathlib import Path

import pandas as pd
from itec632 import get_client as make_llm_client, MODELS, NO_THINK, strip_think   # renamed: A1 defines a tool called get_client
from itec632.corpus import db_path
from itec632.grading import LabChecker

client = make_llm_client()
ck = LabChecker("lab09", part="cluster")
ckb = LabChecker("lab09", part="bonus")   # Part B stretch: summarized in the final report
PACE_S = 1.0

available = {m.id for m in client.models.list().data}
for alias in ("main", "gemma", "small"):
    print(f"{MODELS[alias]:12} {'OK' if MODELS[alias] in available else 'MISSING - check your VPN / key'}")
print("Database:", db_path())

## Part A - Guided practice

### A1. Tools are just functions (with guardrails)

The chapter describes tools as what lets an agent **perceive** its environment (read actions, like a SQL query) and **act** on it (write actions). Our tools are plain Python functions over the Harbor PSA extract in `larkspur.db`. Two habits make them safe enough to hand to a model:

- **Read-only connection.** `mode=ro` means SQLite itself refuses any INSERT/UPDATE/DELETE (the SQL commands that add, change, or remove rows), even if a bug (or the model) tried one.
- **Parameterized queries.** Values go in `params`, never pasted into the SQL text. Each `?` in the SQL is a placeholder that the database fills in safely from `params`, so a strange value can't change what the query does. The model controls the arguments (the values passed to a tool), so treat them like user input.

Tools also return **small, structured** results (at most 20 rows plus a total count). Every token a tool returns is re-sent to the model on every later step of the loop, so verbose tools make agents slow and expensive.

Run the cell below: it defines three tools for you (`lookup_ticket`, `search_tickets`, `list_assets`).

In [ ]:
DB = db_path()
CATEGORIES = ["Account/Access", "Email/M365", "Hardware", "Printing/Scanning", "Network/Internet", "Software",
              "Backup/Restore", "Onboarding/Offboarding", "Phone/VoIP", "Security Report",
              "Server/Infrastructure", "General Request"]
ASSET_TYPES = ["laptop", "desktop", "server", "firewall", "switch", "access_point", "printer", "phone", "nas"]


def ro_connect():
    """Open larkspur.db READ-ONLY. Any write raises sqlite3.OperationalError."""
    con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)
    con.row_factory = sqlite3.Row
    return con


def rows(sql, params=()):
    """Run one parameterized query and return a list of dicts."""
    con = ro_connect()
    try:
        return [dict(r) for r in con.execute(sql, params).fetchall()]
    finally:
        con.close()


class ToolError(Exception):
    """A problem the model can fix (not found, bad value). Its message is sent back to the model."""


def describe_minutes(minutes, clock):
    """240, 'business' -> '4 business hours';  15, '24x7' -> '15 minutes (24x7)'."""
    if clock == "business":
        if minutes % 660 == 0:
            return f"{minutes // 660} business day{'s' if minutes > 660 else ''}"
        if minutes % 60 == 0:
            return f"{minutes // 60} business hour{'s' if minutes > 60 else ''}"
        return f"{minutes} business minutes"
    return f"{minutes // 60} hour{'s' if minutes > 60 else ''} (24x7)" if minutes % 60 == 0 else f"{minutes} minutes (24x7)"


def check_client_code(code):
    code = (code or "").strip().upper()
    if not rows("SELECT 1 FROM clients WHERE code = ?", (code,)):
        raise ToolError(f"No client with code {code!r}. Call get_client with the client's name to find its code.")
    return code


def lookup_ticket(ticket_id):
    """One ticket with its client name/tier and assigned technician."""
    found = rows(
        "SELECT t.ticket_id, t.created_at, t.client_code, c.name AS client_name, c.tier AS client_tier, "
        "t.subject, substr(t.description, 1, 300) AS description, t.category, t.priority, t.status, "
        "tech.name AS assigned_technician, t.first_response_at, t.resolved_at, "
        "substr(t.resolution_notes, 1, 300) AS resolution_notes "
        "FROM tickets t JOIN clients c ON c.code = t.client_code "
        "LEFT JOIN technicians tech ON tech.tech_id = t.assigned_tech_id WHERE t.ticket_id = ?",
        (ticket_id.strip().upper(),))
    if not found:
        raise ToolError(f"Ticket {ticket_id} was not found. Check the ID or use search_tickets.")
    return found[0]


def search_tickets(client_code=None, status=None, category=None, since=None):
    """Total matches + up to 20 tickets, newest first. status='not_closed' = open, in_progress or waiting_on_client."""
    where, params = [], []
    if client_code:
        where.append("client_code = ?"); params.append(check_client_code(client_code))
    if status == "not_closed":
        where.append("status != 'closed'")
    elif status:
        where.append("status = ?"); params.append(status)
    if category:
        where.append("category = ?"); params.append(category)
    if since:
        where.append("created_at >= ?"); params.append(since)
    w = (" WHERE " + " AND ".join(where)) if where else ""
    total = rows(f"SELECT count(*) AS n FROM tickets{w}", params)[0]["n"]
    found = rows(f"SELECT ticket_id, created_at, client_code, status, priority, category, subject "
                 f"FROM tickets{w} ORDER BY created_at DESC LIMIT 20", params)
    return {"total_matches": total, "returned": len(found), "tickets": found}


def list_assets(client_code, type=None, os_contains=None, include_retired=False):
    """Total matches + up to 20 assets for one client. Retired assets excluded unless include_retired."""
    where, params = ["client_code = ?"], [check_client_code(client_code)]
    if type:
        where.append("type = ?"); params.append(type)
    if os_contains:
        where.append("lower(os) LIKE lower(?)"); params.append(f"%{os_contains}%")
    if not include_retired:
        where.append("status != 'retired'")
    w = " WHERE " + " AND ".join(where)
    total = rows(f"SELECT count(*) AS n FROM assets{w}", params)[0]["n"]
    found = rows(f"SELECT asset_tag, hostname, type, model, os, status, notes FROM assets{w} "
                 f"ORDER BY asset_tag LIMIT 20", params)
    return {"total_matches": total, "returned": len(found), "assets": found}


print(json.dumps(lookup_ticket("TCK-119947"), indent=1)[:600])
print(search_tickets(client_code="KVA", status="not_closed"))

**TODO (A1):** write the other two tools. Both are one or two parameterized queries.

This is the first SQL you write yourself. Copy the pattern from the tools above: `check_client_code` and `lookup_ticket` both call `rows(sql, params)` with a `?` in the SQL text and the value in a tuple. `rows()` returns a list of matching rows as dicts (an empty list if none match). `CLIENT_COLS` (given) is the list of columns to select from `clients`.

- `get_client(code)` accepts a 3-letter code **or** part of a name (the model often knows "Kaw Valley Auto Group" but not "KVA").
  1. Try the exact code first, ignoring capitals (SQL's `upper()` or `lower()` on both sides of the comparison).
  2. If nothing matches, search the `name` column with SQL `LIKE`. `LIKE` matches text patterns, and `%` stands for "anything", so the pattern `%Kaw Valley%` matches any name containing "Kaw Valley". Build that pattern from `code` in Python and pass it as the parameter, and use `lower()` on both sides so capitals don't matter.
- `sla_target(tier, priority)` reads the one row of the `sla_targets` table for that tier and priority (columns: `tier, priority, response_minutes, response_clock, resolution_business_minutes, resolution_clock`) and adds readable text with `describe_minutes(minutes, clock)`. Raise `ToolError` if there is no row. The dict you return has these keys and sources:

  | key | value |
  |---|---|
  | `tier`, `priority` | the arguments |
  | `response_minutes`, `response_clock` | the row's columns of the same name |
  | `response_target` | `describe_minutes()` of the row's response minutes and response clock |
  | `resolution_target` | `describe_minutes()` of the row's resolution business minutes and resolution clock |

In [ ]:
CLIENT_COLS = "code, name, industry, city, users, tier, client_since, sites, backup_included, has_voip"


def get_client(code):
    """Client profile by code ('KVA') or part of the name ('Kaw Valley')."""
    code = code.strip()
    found = []
    # TODO: look up the client by code, ignoring capitalization: select CLIENT_COLS from the clients table with rows() and a ? parameter.
    # TODO: if nothing matched, try a partial, case-insensitive match on the client name (SQL LIKE with % wildcards).

    if not found:
        raise ToolError(f"No client matches {code!r}. Try a shorter part of the name.")
    if len(found) > 1:
        return {"matches": [{"code": r["code"], "name": r["name"]} for r in found],
                "note": "Several clients matched; call get_client again with the exact code."}
    return found[0]


def sla_target(tier, priority):
    """Current SLA targets for one tier + priority."""
    # TODO: query sla_targets WHERE tier = ? AND priority = ?; raise ToolError if no row.
    # Return a dict with: tier, priority, response_minutes, response_clock,
    #   response_target (use describe_minutes), resolution_target (use describe_minutes)
    return None

In [ ]:
def safe(fn, *args, **kwargs):
    """Call fn and return its result, or None if it raised (so checks never crash the notebook)."""
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"  {fn.__name__} raised {type(e).__name__}: {e}")
        return None

kva = safe(get_client, "KVA") or {}
bfl = safe(get_client, "Bluffline") or {}
sla = safe(sla_target, "Professional", "P3") or {}
print(kva, bfl, sla, sep="\n")

ck.check("A1_get_client_by_code", kva.get("tier") == "Professional" and kva.get("users") == 72, 2,
         "get_client('KVA') should return Kaw Valley Auto Group (Professional, 72 users). Check the first query.")
ck.check("A1_get_client_by_name", bfl.get("code") == "BFL" and bfl.get("tier") == "Premier", 1,
         "get_client('Bluffline') should fall back to the name search (LIKE with % on both sides).")
ck.check("A1_sla_target", sla.get("response_minutes") == 240 and sla.get("response_target") == "4 business hours", 2,
         "sla_target('Professional','P3') should give response_minutes 240 and response_target '4 business hours'.")

try:
    con = ro_connect()
    con.execute("UPDATE tickets SET status = 'closed' WHERE ticket_id = 'TCK-119947'")
    read_only = False
except sqlite3.OperationalError as e:
    print("Write blocked as expected:", e)
    read_only = True
ck.check("A1_read_only_connection", read_only, 1, "ro_connect() must open the database with mode=ro.")

### A2. Tool schemas and native function calling

The model never sees your Python code. It sees a **tool inventory**: names, descriptions and JSON-schema parameters (the chapter's "function calling" section shows the same idea). A **JSON schema** is a description of what valid arguments look like: which fields exist, their types, and which values are allowed. The model then replies with *which* function to call and *with what arguments* as JSON; your code runs it. Descriptions matter: they are the only documentation the model gets. Notice the `search_tickets` status description below: in everyday speech "open" means "not closed", but the database has three not-closed statuses.

Two schema features do real work for us:
- `enum` lists the only valid values (the model can't invent a tier called "Gold").
- `additionalProperties: false` (no arguments other than the listed ones) and `pattern` (a regular expression, a text pattern such as `^TCK-\d{6}$`, meaning "TCK- followed by exactly 6 digits") let us reject made-up arguments before running anything.

In [ ]:
def tool_schema(name, description, properties, required):
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object", "properties": properties, "required": required,
                       "additionalProperties": False}}}

TOOL_SCHEMAS = {
    "lookup_ticket": tool_schema(
        "lookup_ticket", "Get one ticket by ID: status, priority, category, client name and tier, assigned technician, timestamps.",
        {"ticket_id": {"type": "string", "pattern": r"^TCK-\d{6}$", "description": "Ticket ID like TCK-119947 (TCK- plus 6 digits)."}},
        ["ticket_id"]),
    "search_tickets": tool_schema(
        "search_tickets", "Search tickets. Returns total_matches and up to 20 tickets, newest first. "
        "Use status='not_closed' for tickets still open in any way (open, in_progress, waiting_on_client).",
        {"client_code": {"type": "string", "pattern": r"^[A-Za-z]{3}$", "description": "3-letter client code like KVA (use get_client to find it)."},
         "status": {"type": "string", "enum": ["open", "in_progress", "waiting_on_client", "closed", "not_closed"]},
         "category": {"type": "string", "enum": CATEGORIES},
         "since": {"type": "string", "pattern": r"^\d{4}-\d{2}-\d{2}$", "description": "Created on/after this date, YYYY-MM-DD."}},
        []),
    "get_client": tool_schema(
        "get_client", "Get a client's profile (code, name, service tier, users, sites). Accepts the 3-letter code or part of the name.",
        {"code": {"type": "string", "minLength": 2, "description": "Client code like KVA or part of the name like 'Kaw Valley'."}},
        ["code"]),
    "list_assets": tool_schema(
        "list_assets", "List one client's managed assets. Returns total_matches and up to 20 assets. Retired assets excluded unless include_retired.",
        {"client_code": {"type": "string", "pattern": r"^[A-Za-z]{3}$", "description": "3-letter client code like EMS."},
         "type": {"type": "string", "enum": ASSET_TYPES},
         "os_contains": {"type": "string", "description": "Filter on OS text, e.g. 'Windows 10'."},
         "include_retired": {"type": "boolean"}},
        ["client_code"]),
}

**TODO (A2):** write the schema for `sla_target`. Use `tool_schema(...)`. Both parameters are required strings with an `enum`: tier is one of `Essentials`, `Professional`, `Premier`; priority is one of `P1`-`P4`. Write a one-sentence description that tells the model what it returns.

In [ ]:
SLA_TARGET_SCHEMA = None   # TODO: tool_schema("sla_target", "...", {...}, ["tier", "priority"])

if SLA_TARGET_SCHEMA:
    TOOL_SCHEMAS["sla_target"] = SLA_TARGET_SCHEMA
TOOLS = list(TOOL_SCHEMAS.values())
print([t["function"]["name"] for t in TOOLS])

Now one native tool call. We send the question **and** the tool inventory. A tool-capable model answers with `tool_calls` (function name + JSON arguments) instead of text. Nothing runs yet: deciding *what* to call is the model's job, running it is ours.

**Reading the output:** `finish_reason` says why the model stopped writing: `tool_calls` means it wants a tool run, `stop` means it gave a normal text answer. `tool calls` lists each requested tool with its arguments as JSON. `text` is usually empty (`None`) when the model calls a tool.

In [ ]:
FUNCTIONS = {"lookup_ticket": lookup_ticket, "search_tickets": search_tickets, "get_client": get_client,
             "list_assets": list_assets, "sla_target": sla_target}

resp = client.chat.completions.create(
    model=MODELS["main"],
    messages=[{"role": "user", "content": "What's the status of ticket TCK-119921?"}],
    tools=TOOLS, extra_body=NO_THINK, temperature=0)
msg = resp.choices[0].message
first_calls = [(tc.function.name, tc.function.arguments) for tc in (msg.tool_calls or [])]
print("finish_reason:", resp.choices[0].finish_reason)
print("tool calls:", first_calls or "(none)")
print("text:", msg.content)

s = (SLA_TARGET_SCHEMA or {}).get("function", {})
props = s.get("parameters", {}).get("properties", {})
ck.check("A2_sla_schema", s.get("name") == "sla_target"
         and set(props.get("tier", {}).get("enum", [])) == {"Essentials", "Professional", "Premier"}
         and set(props.get("priority", {}).get("enum", [])) == {"P1", "P2", "P3", "P4"}
         and set(s.get("parameters", {}).get("required", [])) == {"tier", "priority"}, 2,
         "The sla_target schema needs name 'sla_target', both enums, and required ['tier', 'priority'].")
ck.check("A2_native_tool_call", len(first_calls) >= 1, 1,
         "The model did not return tool_calls. Did you pass tools=TOOLS? Is the model name right?")

### A3. Validate arguments and send errors back to the model

Under "Planning failures", the chapter lists the most common kind, tool use failures: an **invalid tool** (a name that doesn't exist), a **valid tool with invalid parameters** (wrong type, missing field), and a **valid tool with incorrect parameter values** (a well-formed but wrong client code). An agent can often recover, but only if the error message tells it what went wrong. So `execute_tool_call` never **raises** an exception (Python's way of stopping with an error): every problem becomes a short JSON `{"error": ...}` that goes back to the model as the tool result.

`validate_args` (given) checks arguments against the schema with `jsonschema`, a Python library that compares a JSON object with a JSON schema and lists every mismatch.

In [ ]:
from jsonschema import Draft202012Validator

WRITE_TOOLS = set()   # filled in A5


def validate_args(name, args):
    """List of readable problems with `args` for tool `name` (empty list = valid)."""
    schema = TOOL_SCHEMAS[name]["function"]["parameters"]
    problems = []
    for e in sorted(Draft202012Validator(schema).iter_errors(args), key=str):
        where = "/".join(map(str, e.path)) or "arguments"
        if e.validator == "pattern":   # regexes confuse models; repeat the human description instead
            problems.append(f"{where}: {e.instance!r} is not in the right format. "
                            + schema["properties"].get(where, {}).get("description", ""))
        else:
            problems.append(f"{where}: {e.message}")
    return problems


def confirm_gate(name, args, confirm):
    """Placeholder until A5: refuse every write."""
    return json.dumps({"error": "Write tools are disabled."})

**TODO (A3):** finish `execute_tool_call`. After the arguments are parsed, call `validate_args(name, args)`. If it returns any problems, **return** (don't raise) an error JSON string, like the other `{"error": ...}` returns in this function, whose message names the tool and lists all the problems (join them into one string, separated by semicolons).

In [ ]:
def execute_tool_call(name, arguments_json, confirm=None):
    """Run one tool call from the model. ALWAYS returns a string for the 'tool' message."""
    if name not in TOOL_SCHEMAS:
        return json.dumps({"error": f"Unknown tool {name!r}. Available tools: {list(TOOL_SCHEMAS)}"})
    try:
        args = json.loads(arguments_json or "{}")
    except json.JSONDecodeError:
        return json.dumps({"error": "Arguments were not valid JSON. Send a JSON object."})
    args = {k: v for k, v in args.items() if v is not None}

    # TODO: validate with validate_args(name, args); if there are problems, return the error JSON

    if name in WRITE_TOOLS:
        return confirm_gate(name, args, confirm)
    try:
        return json.dumps(FUNCTIONS[name](**args), default=str)[:4000]
    except ToolError as e:
        return json.dumps({"error": str(e)})
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})

In [ ]:
tests = {
    "bad_format": execute_tool_call("lookup_ticket", '{"ticket_id": "119947"}'),
    "extra_arg": execute_tool_call("get_client", '{"code": "KVA", "region": "KS"}'),
    "bad_enum": execute_tool_call("search_tickets", '{"client_code": "KVA", "status": "pending"}'),
    "wrong_value": execute_tool_call("search_tickets", '{"client_code": "XYZ"}'),
    "unknown_tool": execute_tool_call("delete_ticket", '{"ticket_id": "TCK-119947"}'),
    "good": execute_tool_call("lookup_ticket", '{"ticket_id": "TCK-119947"}'),
}
for k, v in tests.items():
    print(f"{k:13} {v[:150]}")

err = lambda s: '"error"' in s
ck.check("A3_rejects_bad_arguments", err(tests["bad_format"]) and err(tests["extra_arg"]) and err(tests["bad_enum"]), 2,
         "Bad format, extra arguments and bad enum values should all return an error JSON. Did you return validate_args problems?")
ck.check("A3_wrong_value_and_unknown_tool", err(tests["wrong_value"]) and err(tests["unknown_tool"]), 1,
         "An unknown client code and an unknown tool name should both come back as error JSON (not exceptions).")
ck.check("A3_good_call_runs", (not err(tests["good"])) and "TCK-119947" in tests["good"], 1,
         "A valid lookup_ticket call should return the ticket JSON. Is validation rejecting valid arguments?")

**Reading the output:** each line is one test call: its name on the left, and on the right the first 150 characters of what would go back to the model. Every line except `good` should be an `{"error": ...}` message that tells the model what to fix.

### A4. The agent loop

An agent is a loop around the model. The chapter's "Reflection and error correction" section describes the ReAct pattern (thought -> act -> observation, repeated until done); here, function calling does the planning one step at a time:

1. Send the conversation plus the tool inventory.
2. If the reply has **no** tool calls, it's the final answer: stop.
3. Otherwise append the assistant message (with its `tool_calls`), run **each** tool call, and append one `{"role": "tool", "tool_call_id": ..., "content": result}` message per call.
4. Repeat, but never more than `max_steps` model calls. An unbounded loop is how agents burn money.

Why this order? The model must see its own request first (the assistant message with `tool_calls`), then each result tagged with that request's `tool_call_id`. Without the id it can't tell which result answers which call when it asked for several at once.

The helpers below handle the API details. `chat()` uses `temperature=0`, which makes the model pick its most likely next token every time, so runs are as repeatable as possible (Lab 03 covers temperature). `trace` records everything we'll evaluate later: steps (model calls), tool calls, tool errors, tokens, and why the loop stopped.

In [ ]:
AGENT_SYSTEM = """You are the Larkspur Ridge IT help-desk data assistant. Today is 2026-11-02.
Answer questions about clients, tickets, assets and SLA targets by calling the tools.
- Never guess an ID, a client code or a number: look it up.
- If a tool returns an error, read it, fix the arguments and try again.
- When you have what you need, answer in 1-4 sentences with the specific IDs, names and numbers."""


def chat(model, messages, tools):
    """One model call (thinking off, deterministic), then a short pause for the rate limit."""
    resp = client.chat.completions.create(model=model, messages=messages, tools=tools,
                                          extra_body=NO_THINK, temperature=0, max_tokens=800)
    time.sleep(PACE_S)
    return resp


def assistant_turn(msg):
    """The assistant message, including its tool calls, as a plain dict to append to `messages`."""
    turn = {"role": "assistant", "content": msg.content or ""}
    if msg.tool_calls:
        turn["tool_calls"] = [{"id": tc.id, "type": "function",
                               "function": {"name": tc.function.name, "arguments": tc.function.arguments or "{}"}}
                              for tc in msg.tool_calls]
    return turn


def deny_all(name, args):
    return False

**TODO (A4):** inside the loop, after the "final answer" check, append the assistant turn (`assistant_turn()` converts the reply) and then, **for each** tool call `tc` in `msg.tool_calls`:
1. run it with `execute_tool_call()`, passing the call's function name, its arguments, and `confirm`;
2. append a tool message: a dict with role `tool`, the call's id as `tool_call_id`, and the result string as `content`;
3. log it in the trace with `record_call()`.

About 6 lines.

In [ ]:
def record_call(trace, tc, result):
    trace["tool_calls"].append({"name": tc.function.name, "args": tc.function.arguments, "result": result[:200]})
    if '"error"' in result:
        trace["tool_errors"] += 1


def run_agent(question, model=MODELS["main"], tools=None, max_steps=6, confirm=deny_all, verbose=False):
    tools = tools or TOOLS
    messages = [{"role": "system", "content": AGENT_SYSTEM}, {"role": "user", "content": question}]
    trace = {"question": question, "model": model, "steps": 0, "tool_calls": [], "tool_errors": 0,
             "prompt_tokens": 0, "completion_tokens": 0, "answer": "", "stop_reason": "max_steps"}
    t0 = time.perf_counter()
    for _ in range(max_steps):
        resp = chat(model, messages, tools)
        trace["steps"] += 1
        if resp.usage:
            trace["prompt_tokens"] += resp.usage.prompt_tokens or 0
            trace["completion_tokens"] += resp.usage.completion_tokens or 0
        msg = resp.choices[0].message
        if not msg.tool_calls:                       # no tool call = final answer
            trace["answer"] = strip_think(msg.content or "")
            trace["stop_reason"] = "answer"
            break
        n_before = len(messages)
        # TODO: add the model's reply to messages (assistant_turn() converts it to the right format).
        # TODO: run each tool call the model asked for (execute_tool_call), add each result to messages as a "tool" message, and log it with record_call().

        if len(messages) == n_before:                # guard: stops early until the TODO above is done
            trace["stop_reason"] = "loop_body_missing"
            break
    trace["latency_s"] = round(time.perf_counter() - t0 - PACE_S * trace["steps"], 2)
    trace["total_tokens"] = trace["prompt_tokens"] + trace["completion_tokens"]
    trace["messages"] = messages
    if verbose:
        for c in trace["tool_calls"]:
            print(f"  -> {c['name']}({c['args']})  =>  {c['result'][:120]}")
        print(f"  [{trace['stop_reason']}, {trace['steps']} steps, {trace['total_tokens']} tokens] {trace['answer'][:300]}")
    return trace

In [ ]:
print("Q: Which open tickets does Kaw Valley Auto Group have and what's their SLA response target?")
demo = run_agent("Which open tickets does Kaw Valley Auto Group have and what's their SLA response target?", verbose=True)

print("\nSame question with max_steps=1:")
short = run_agent("Which open tickets does Kaw Valley Auto Group have and what's their SLA response target?",
                  max_steps=1, verbose=True)

ck.check("A4_loop_runs_tools", len(demo["tool_calls"]) >= 1 and demo["stop_reason"] == "answer" and demo["answer"], 3,
         "run_agent should call at least one tool and finish with stop_reason 'answer'. Did you append the tool messages?")
ck.check("A4_max_steps_respected", short["steps"] <= 1, 1, "The loop must stop after max_steps model calls.")
ck.record("demo_steps", demo["steps"])
ck.record("demo_tokens", demo["total_tokens"])

**Reading the output:** each `->` line is one tool call the model made: the tool name, its arguments, and (after `=>`) the start of the result. The last line in brackets shows why the loop stopped (`answer` = the model gave a final answer, `max_steps` = it ran out of steps, `loop_body_missing` = the TODO isn't done yet), how many model calls it took, and the total tokens, followed by the start of the answer.

### A5. Write actions need a human in the loop (simulated)

The chapter is direct about this: read actions let an agent *perceive*, **write actions** let it change the world, and that is where the risk is. A common middle ground is to let the agent *propose* a write and require a human to confirm it. Our "write" tool, `request_ticket_update`, never touches the database (the connection is read-only anyway). It only records what *would* change, and only after `confirm(name, args)` returns `True`.

**TODO (A5):** write `confirm_gate`. If `confirm(name, args)` is falsy (`False`, `None`, `0`, or empty; `not x` is `True` for these) or `confirm` is `None`, return an error JSON that says the user **declined** and the agent should not retry. Otherwise append `args` to `WRITE_LOG` and return a JSON string with two keys: `simulated` (true) and `would_update` (the args), so the model knows the change was only simulated.

In [ ]:
WRITE_LOG = []
TOOL_SCHEMAS["request_ticket_update"] = tool_schema(
    "request_ticket_update",
    "Ask to change a ticket's status. A human must approve; nothing changes without approval.",
    {"ticket_id": {"type": "string", "pattern": r"^TCK-\d{6}$", "description": "Ticket ID like TCK-119947."},
     "new_status": {"type": "string", "enum": ["open", "in_progress", "waiting_on_client", "closed"]},
     "note": {"type": "string", "description": "Why the change is needed."}},
    ["ticket_id", "new_status", "note"])
WRITE_TOOLS = {"request_ticket_update"}


def confirm_gate(name, args, confirm):
    """Only 'perform' a write (simulated) if a human confirmed it."""
    # TODO: if there is no confirm function, or the user says no, return an error JSON that tells the model the user declined.
    # TODO: otherwise record args in WRITE_LOG and return a JSON result that says the change was simulated, not made.
    return json.dumps({"error": "not implemented"})


approve_all = lambda name, args: True

In [ ]:
req = json.dumps({"ticket_id": "TCK-119947", "new_status": "closed", "note": "User confirmed password works."})
denied = execute_tool_call("request_ticket_update", req, confirm=deny_all)
log_after_deny = len(WRITE_LOG)
approved = execute_tool_call("request_ticket_update", req, confirm=approve_all)
print("denied:  ", denied)
print("approved:", approved)
status_now = rows("SELECT status FROM tickets WHERE ticket_id = 'TCK-119947'")[0]["status"]
print("database status is still:", status_now)

ck.check("A5_write_needs_confirmation", "declin" in denied.lower() and log_after_deny == 0, 2,
         "With deny_all the gate must return an error that says 'declined' and must not add to WRITE_LOG.")
ck.check("A5_simulated_write_only", '"simulated": true' in approved and status_now == "in_progress", 1,
         "With approve_all the gate returns {'simulated': true, ...}; the database itself never changes.")

print("\nAgent asked to close a ticket, with a human who says no:")
WRITE_DEMO_TOOLS = TOOLS + [TOOL_SCHEMAS["request_ticket_update"]]
w = run_agent("Please close ticket TCK-119947, the user says the new password works.",
              tools=WRITE_DEMO_TOOLS, confirm=deny_all, verbose=True)
ck.record("write_tool_attempted", any(c["name"] == "request_ticket_update" for c in w["tool_calls"]))

### A6. Evaluate the agent on 15 tasks with known answers

You can't improve what you don't measure (Ch. 3-4), and agents are no exception. Because our tools sit on a database, we can compute the **correct answer for every task straight from SQL**, independent of the model. Each task lists the facts a correct answer must contain (`must_include`): a list of groups, where each group is a list of acceptable spellings.

`min_steps` is the fewest model calls a sensible plan needs (tool rounds + the final answer). Comparing it with the actual steps measures **efficiency**, one of the chapter's agent evaluation dimensions.

You don't need to read the next cell in detail: it builds the 15 tasks with SQL. To see what one task looks like, run `TASKS[3]` in a new cell.

In [ ]:
def q(sql, *params):
    return rows(sql, params)

def phrases(minutes, clock):
    """Acceptable ways to write an SLA target in an answer."""
    p = {describe_minutes(minutes, clock), f"{minutes} min"}
    if minutes % 60 == 0:
        h = minutes // 60
        p |= {f"{h} hour", f"{h}-hour", f"{h} business hour"} if clock == "business" else {f"{h} hour", f"{h}-hour"}
    return sorted(p)

WORDS = "zero one two three four five six seven eight nine ten eleven twelve".split()

def count_phrases(n, noun):
    """'3 server', 'three server' ... so a bare digit elsewhere in the answer doesn't count."""
    p = [f"{n} {noun}", f"{n} {noun.capitalize()}"]
    if n < len(WORDS):
        p += [f"{WORDS[n]} {noun}", f"{WORDS[n]} {noun.capitalize()}"]
    if n == 1:
        p += ["only one", "a single", "just one"]
    return p

def sla_row(tier, prio):
    return q("SELECT * FROM sla_targets WHERE tier = ? AND priority = ?", tier, prio)[0]

TASKS = []
def task(tid, question, must_include, min_steps, kind):
    TASKS.append({"id": tid, "question": question, "must_include": must_include, "min_steps": min_steps, "kind": kind})

t = q("SELECT status, priority FROM tickets WHERE ticket_id = 'TCK-119956'")[0]
task("T01", "What is the status and priority of ticket TCK-119956?", [[t["status"]], [t["priority"]]], 2, "single")

tier = q("SELECT tier FROM clients WHERE name = 'Bluffline Logistics'")[0]["tier"]
task("T02", "Which service tier is Bluffline Logistics on?", [[tier]], 2, "single")

r = sla_row("Essentials", "P2")
task("T03", "What is the first-response SLA target for a P2 ticket from an Essentials client?",
     [phrases(r["response_minutes"], r["response_clock"])], 2, "single")

kva_tier = q("SELECT tier FROM clients WHERE code = 'KVA'")[0]["tier"]
kva_open = q("SELECT ticket_id, priority FROM tickets WHERE client_code = 'KVA' AND status != 'closed'")
r = sla_row(kva_tier, kva_open[0]["priority"])
task("T04", "Which open tickets does Kaw Valley Auto Group have and what's their SLA response target?",
     [[x["ticket_id"]] for x in kva_open] + [phrases(r["response_minutes"], r["response_clock"])], 4, "multi")

srv = q("SELECT hostname FROM assets WHERE client_code = 'EMS' AND type = 'server' AND status != 'retired'")
task("T05", "How many servers does Emberline Metalworks have, and what are their hostnames?",
     [count_phrases(len(srv), "server")] + [[s["hostname"]] for s in srv], 3, "multi")

tech = q("SELECT tech.name FROM tickets t JOIN technicians tech ON tech.tech_id = t.assigned_tech_id "
         "WHERE t.ticket_id = 'TCK-119921'")[0]["name"]
task("T06", "Who is the assigned technician on ticket TCK-119921?", [[tech]], 2, "single")

n = q("SELECT count(*) AS n FROM tickets WHERE client_code = 'DPA' AND category = 'Email/M365' "
      "AND created_at >= '2026-06-01'")[0]["n"]
task("T07", "How many Email/M365 tickets has Dunmore & Pace Attorneys had since 2026-06-01?",
     [count_phrases(n, "ticket") + count_phrases(n, "email")], 3, "multi")

w10 = q("SELECT asset_tag, hostname FROM assets WHERE client_code = 'KVA' AND os LIKE 'Windows 10%' AND status != 'retired'")
task("T08", "Which Windows 10 machines does Kaw Valley Auto Group still have in service?",
     [[a["asset_tag"], a["hostname"]] for a in w10], 3, "multi")

t = q("SELECT t.priority, c.name, c.tier FROM tickets t JOIN clients c ON c.code = t.client_code "
      "WHERE ticket_id = 'TCK-119940'")[0]
r = sla_row(t["tier"], t["priority"])
task("T09", "Ticket TCK-119940: which client is it for, what tier are they, and what is the response target for that ticket's priority?",
     [[t["name"].split()[0]], [t["tier"]], phrases(r["response_minutes"], r["response_clock"])], 3, "multi")

r = sla_row("Premier", "P1")
task("T10", "What is the P1 response target for Premier clients, and does that clock run 24x7 or only in business hours?",
     [phrases(r["response_minutes"], r["response_clock"]), ["24x7", "24/7", "24 x 7", "around the clock", "24 hours"]], 2, "single")

fw = q("SELECT asset_tag FROM assets WHERE client_code = 'AMS' AND type = 'firewall' AND status != 'retired'")
task("T11", "List the firewalls at Arbor Mill Storage (asset tags).", [[f["asset_tag"]] for f in fw], 3, "multi")

n = q("SELECT count(*) AS n FROM tickets WHERE client_code = 'CDV' AND status != 'closed'")[0]["n"]
assert n == 0
task("T12", "Does Cedar Valley Veterinary have any tickets that are not closed?",
     [["no open", "no tickets", "none", "no non-closed", "not have any", "doesn't have any", "does not have any",
       "no unclosed", "zero", "0 open", "are all closed", "all closed", "no active"]], 3, "multi")

u = {r["code"]: r["users"] for r in q("SELECT code, users FROM clients WHERE code IN ('KVA', 'EMS')")}
d = abs(u["EMS"] - u["KVA"])
task_diff = [f"{d} more", f"{d} users", f"by {d}", f"{d} managed", f"difference of {d}", f"{d} additional", f"{d} fewer"]
task("T13", "Which has more managed users, Kaw Valley Auto Group or Emberline Metalworks, and by how many?",
     [["Emberline" if u["EMS"] > u["KVA"] else "Kaw Valley"], task_diff], 2, "multi")

t = q("SELECT priority, resolved_at FROM tickets WHERE ticket_id = 'TCK-118742'")[0]
hhmm = t["resolved_at"][-5:]
task("T14", "What priority was ticket TCK-118742 and when was it resolved?",
     [[t["priority"]], [hhmm, "1:00 pm", "1:00 p.m.", "1 pm", "1pm"]], 2, "single")

task("T15", "Look up ticket TCK-99999 and tell me its status.",
     [["not found", "invalid", "doesn't exist", "does not exist", "no ticket", "couldn't find", "could not find",
       "not a valid", "unable to find", "incorrect format", "wrong format", "not in the correct format", "6 digits", "six digits"]],
     2, "error-recovery")

pd.DataFrame(TASKS)[["id", "kind", "min_steps", "question"]]

**Reading the table:** one row per task. `kind` is `single` (one lookup answers it), `multi` (the agent must chain two or more tools, using one result to make the next call), or `error-recovery` (the question is built to trigger a tool error the agent must handle). `min_steps` is the fewest model calls a sensible plan needs.

**The scorer (provided).** `score_answer(answer, must_include)` returns `True` only if **every** group in `must_include` has **at least one** spelling that appears in the answer. It compares case-insensitively and treats `_` as a space (so `in_progress` matches "in progress").

**TODO (A6, predict, then check):** an automatic scorer is only as good as its spelling lists, and you'll have to judge its verdicts in A7. Before you run the cell, predict `True` or `False` for the two example answers in `MY_PREDICTIONS`. The cell then prints what the scorer really says. If you were wrong, work out why before moving on.

In [ ]:
def norm(s):
    return str(s).lower().replace("_", " ")


def score_answer(answer, must_include):
    """True if every group has at least one acceptable spelling in the answer (provided)."""
    return all(any(norm(alt) in norm(answer) for alt in group) for group in must_include)


SCORER_EXAMPLES = {
    "a": ("TCK-119947 is In_Progress.", [["TCK-119947"], ["in progress"]]),
    "b": ("TCK-119947 is in progress; the response target is 4 hours.", [["TCK-119947"], ["4 business hours", "240 min"]]),
}
MY_PREDICTIONS = {"a": None, "b": None}   # TODO: before running, replace each None with True or False

for key, (ans, must) in SCORER_EXAMPLES.items():
    print(f"{key}: scorer says {score_answer(ans, must)}, you predicted {MY_PREDICTIONS[key]}  <- {ans!r}")

Now run all 15 tasks on `qwen-moe` (about 45-60 requests, 2-4 minutes). The table shows, per task: pass/fail, steps vs. the minimum, tool calls, tool errors, and tokens.

In [ ]:
def eval_tasks(tasks, model=MODELS["main"], max_steps=6):
    out, traces = [], {}
    for tk in tasks:
        tr = run_agent(tk["question"], model=model, max_steps=max_steps)
        traces[tk["id"]] = tr
        out.append({"task": tk["id"], "kind": tk["kind"], "model": model,
                    "passed": score_answer(tr["answer"], tk["must_include"]),
                    "steps": tr["steps"], "min_steps": tk["min_steps"], "extra_steps": tr["steps"] - tk["min_steps"],
                    "tool_calls": len(tr["tool_calls"]), "tool_errors": tr["tool_errors"],
                    "prompt_tokens": tr["prompt_tokens"], "completion_tokens": tr["completion_tokens"],
                    "total_tokens": tr["total_tokens"], "latency_s": tr["latency_s"], "stop": tr["stop_reason"]})
        print(f"{tk['id']} {'PASS' if out[-1]['passed'] else 'fail'}  steps={tr['steps']}  tokens={tr['total_tokens']}")
    return pd.DataFrame(out), traces

results, traces = eval_tasks(TASKS)
success = results["passed"].mean()
print(f"\nSuccess rate: {success:.0%}   mean steps: {results['steps'].mean():.2f}   "
      f"mean tokens/task: {results['total_tokens'].mean():.0f}   tool errors: {results['tool_errors'].sum()}")
display(results)
print(results.groupby("kind")[["passed", "steps", "extra_steps", "total_tokens"]].mean().round(2))

ck.check("A6_ran_all_tasks", len(results) == 15 and results["steps"].min() >= 1 and results["tool_calls"].sum() > 0, 2,
         "All 15 tasks should run through run_agent and make tool calls. Finish the A4 loop body first.")
ck.check("A6_scorer_predictions", all(MY_PREDICTIONS.get(k) is score_answer(*v) for k, v in SCORER_EXAMPLES.items()), 1,
         "Set MY_PREDICTIONS to True/False to match what the scorer printed for each example, and note why example b comes out as it does.")
if success < 0.6:   # recorded, not graded: the model's day-to-day behavior is not under your control
    print("Note: qwen-moe usually solves 10+ of 15. If your A3/A4 checks pass, explain the shortfall in Q1 using the A7 traces.")
ck.record("success_rate_qwen_moe", float(success))
ck.record("mean_steps", float(results["steps"].mean()))
ck.record("mean_extra_steps", float(results["extra_steps"].mean()))
ck.record("mean_tokens_per_task", float(results["total_tokens"].mean()))
ck.record("total_tool_errors", int(results["tool_errors"].sum()))

**Reading the tables:** the big table has one row per task. `passed` is whether the answer contained every required fact. `steps` is how many model calls the agent made, `min_steps` the fewest needed, and `extra_steps` the difference: positive means wasted steps, negative means it answered before doing the lookups it needed (a problem, not a win). `tool_calls` and `tool_errors` count all tool calls and the ones that came back as errors. `prompt_tokens` adds up everything sent to the model over all steps, `completion_tokens` what the model wrote, and `total_tokens` both. `latency_s` is seconds for the task (pauses not counted), and `stop` is why the loop ended. The small table below it averages by `kind`; the average of `passed` is the success rate (1.0 = all passed).

### A7. Failure modes: planning, tool misuse, efficiency

The chapter's "Agent Failure Modes and Evaluation" section sorts problems into three groups: planning failures, tool failures, and efficiency. We split the first group in two so you can count tool-use errors separately:
- **Planning failures, tool use** (tag `tool_misuse`): the chapter calls this the most common planning failure: an invalid tool, a valid tool with invalid parameters, or a valid tool with incorrect parameter values (e.g. `status="open"` when the user meant "not closed").
- **Planning failures, goal** (tag `planning`): the plan doesn't reach the goal or breaks a constraint (missing step, answering before the data is in, giving up, hitting the step limit), or a reflection error (the agent says it's done when it isn't).
- **Tool failures** (tag `tool_failure`): the right tool was called correctly but returned a wrong output, or the agent didn't have the tool it needed. Our tools are tested SQL, so these should be rare.
- **Efficiency** (tag `efficiency`): the agent gets there, but with more steps, tokens or time than needed.

`auto_tag` makes a first guess from the trace. Then **you** read the traces and decide, because automatic tags are often wrong about *why*. Two common cases:
- **T15's tool error is planned.** TCK-99999 has only 5 digits, so the tool is supposed to reject it. `auto_tag` still says `tool_misuse`; judge whether the agent *recovered* (told the user the ID is invalid). If it did, label it `none`.
- **The scorer can be too strict.** If the answer is right but worded differently from every accepted spelling (for example "4 hours" where "4 business hours" was required), label it `none: scorer too strict because ...`.

**Reading the output:** the table lists the tasks worth a closer look, with `auto_tags` holding the first-guess categories. Below it, each trace shows the question, then every tool call (`->` with its arguments, `<=` the start of the result), then the final answer with the stop reason and step count in brackets.

In [ ]:
def auto_tag(row, tr):
    tags = []
    if row["tool_errors"] > 0:
        tags.append("tool_misuse")
    if row["stop"] == "max_steps" or (not row["passed"] and row["tool_errors"] == 0):
        tags.append("planning")
    calls = [(c["name"], c["args"]) for c in tr["tool_calls"]]
    if row["extra_steps"] > 0 or len(calls) != len(set(calls)):
        tags.append("efficiency")
    return tags


def show_trace(tid, traces=traces):
    tr = traces[tid]
    print(f"{tid}: {tr['question']}")
    for c in tr["tool_calls"]:
        print(f"  -> {c['name']}({c['args']})\n     <= {c['result'][:160]}")
    print(f"  ANSWER [{tr['stop_reason']}, {tr['steps']} steps]: {tr['answer'][:400]}\n")


results["auto_tags"] = [auto_tag(r, traces[r["task"]]) for _, r in results.iterrows()]
candidates = results[(~results["passed"]) | (results["auto_tags"].str.len() > 0)]
display(candidates[["task", "passed", "steps", "min_steps", "tool_errors", "auto_tags"]])
for tid in candidates["task"].head(5):
    show_trace(tid)

**TODO (A7):** read the traces above (use `show_trace("T04")` for any task). Label **at least 3** tasks (all of them if fewer than 3 failed or were inefficient; if every task was perfect, pick the 3 longest traces and explain what *could* go wrong). Write `"task_id": "category: what happened, in your words"`, with category one of planning / tool_misuse / tool_failure / efficiency / none.

In [ ]:
MY_LABELS = {
    # "T04": "planning: searched with status='open' and missed the in_progress ticket, then answered 'no open tickets'",
}

In [ ]:
need = min(3, max(1, len(candidates)))
valid = {k: v for k, v in MY_LABELS.items()
         if k in set(results["task"]) and len(str(v)) >= 20
         and str(v).split(":")[0].strip() in {"planning", "tool_misuse", "tool_failure", "efficiency", "none"}}
ck.check("A7_failure_labels", len(valid) >= need, 2,
         f"Label at least {need} task(s) as 'category: explanation' (category: planning / tool_misuse / tool_failure / efficiency / none).")
ck.record("n_failure_candidates", int(len(candidates)))
ck.record("my_label_categories", sorted({str(v).split(':')[0].strip() for v in valid.values()}))

### A8. Does the model matter? qwen-moe vs gemma-moe vs qwen-small

Tool use is a skill models learn in post-training, and it varies a lot between models and sizes. Run the same 5 tasks (a mix of single and multi-step) on three models and compare success, steps, tool errors and tokens. About 40 requests.

**TODO (A8):** loop over `COMPARE_MODELS`. For each model, run `eval_tasks()` on `COMPARE_TASKS` with that model; it returns a results table and the traces. Append the table to `frames` and store the traces in `compare_traces` under the model name. The concatenation into `compare` is done for you.

In [ ]:
COMPARE_MODELS = [MODELS["main"], MODELS["gemma"], MODELS["small"]]
COMPARE_TASKS = [tk for tk in TASKS if tk["id"] in ("T01", "T04", "T06", "T09", "T13")]
frames, compare_traces = [], {}

# TODO: run eval_tasks() on COMPARE_TASKS once per model in COMPARE_MODELS; keep each results table in frames and each model's traces in compare_traces.

compare = pd.concat(frames) if frames else pd.DataFrame(columns=["model", "task", "passed", "steps", "tool_errors", "total_tokens"])

In [ ]:
if len(compare):
    summary = compare.groupby("model").agg(success=("passed", "mean"), mean_steps=("steps", "mean"),
                                           tool_errors=("tool_errors", "sum"), mean_tokens=("total_tokens", "mean"),
                                           mean_latency_s=("latency_s", "mean")).round(2)
    display(summary)
    display(compare.pivot(index="task", columns="model", values="passed"))
    for m, row in summary.iterrows():
        ck.record(f"success_{m}", float(row["success"]))
        ck.record(f"tool_errors_{m}", int(row["tool_errors"]))
    small = MODELS["small"]
    failed_small = compare[(compare["model"] == small) & (~compare["passed"])]["task"].tolist()
    if small in compare_traces:      # for Q3: one trace from the small model (change the task id to see others)
        print(f"\n{small} failed: {failed_small or 'none'}. One of its traces:")
        show_trace((failed_small or ["T09"])[0], compare_traces[small])
ck.check("A8_three_models_compared", len(compare) == 15 and compare["model"].nunique() == 3, 2,
         "Run the 5 comparison tasks on all three models (15 rows).")
ck.check("A8_summary_recorded", any(k.startswith("success_gemma") for k in ck.records), 1,
         "Run the summary cell after the comparison so the success rates are recorded.")

**Reading the tables:** the first table has one row per model. `success` is the fraction of the 5 tasks passed (0.6 = 3 of 5), `mean_steps` the average number of model calls, `tool_errors` the total tool errors, `mean_tokens` the average tokens per task, and `mean_latency_s` the average seconds per task. The second table shows pass (`True`) or fail (`False`) for every task and model, so you can see exactly where a model broke down. Under the tables is one trace from `qwen-small`. To read any other one, run `show_trace("T09", compare_traces[MODELS["small"]])` with the task id and model you want.

## Part B - Optional stretch (bonus, up to 10 points): Co. app v3

Your Co. app (Lab 8, v2) answers from documents. Version 3 adds **tools** and a **router** in `assistant.answer()`:

| route | when | what answer() does |
|---|---|---|
| `docs` | policies, prices, procedures | retrieve chunks, one model call (v2 behavior) |
| `data` | tickets, clients, assets | no retrieval; agent loop with the SQLite tools |
| `both` | a question that needs both | retrieve chunks **and** offer tools |

Steps: (B1) copy your v2 into `coapp/`, (B2) write `coapp/tools.py` from your Part A code, (B3) update `coapp/prompts.py` for tools and write the new `assistant.py` with a `route()` function, (B4) run the bonus checks. Stuck? `coapp_reference/` in this folder is a complete v3 you can read or copy (Lab 14 builds on v3).

In [ ]:
import importlib, shutil, sys

def find_lab_dir(slug, must="coapp_reference"):
    """labs/<slug> (containing `must`) next to this notebook, or in a course repo checked out nearby."""
    here = Path.cwd().resolve()
    for p in [here.parent, *[q / "itec632-course" / "labs" for q in here.parents]]:
        if (p / slug / must).exists():
            return p / slug
    return None

LAB08 = find_lab_dir("lab08-rag")
LAB09 = find_lab_dir("lab09-agents") or (Path.cwd() if Path("coapp_reference").is_dir() else None)
MY_LAB8_COAPP = None          # optional: path to YOUR Lab 8 coapp folder, e.g. Path("../lab08-rag/coapp")
src = Path(MY_LAB8_COAPP) if MY_LAB8_COAPP else (LAB08 / "coapp_reference" if LAB08 and (LAB08 / "coapp_reference").is_dir() else None)

work = Path("coapp")
work.mkdir(exist_ok=True)
only = None                      # None = copy every .py file from src
if src is None and LAB09:        # no Lab 8 folder found: use the v2 parts (config, prompts, retrieve) of the v3 reference
    src, only = LAB09 / "coapp_reference", ["__init__.py", "config.py", "prompts.py", "retrieve.py", "_docs.py"]
    print("No Lab 8 coapp found; using the v2 parts of coapp_reference")
copied = []
if src is not None:
    for f in sorted(src.glob("*.py")):
        if (only is None or f.name in only) and not (work / f.name).exists():   # never overwrites your files
            shutil.copy(f, work / f.name)
            copied.append(f.name)
print("Copied from", src, ":", copied or "nothing new (coapp/ already has these files)")
print("coapp/ contains:", sorted(p.name for p in work.glob("*.py")))

**B2 (TODO):** `coapp/tools.py`. The file must define `TOOLS` (the 5 read-only schemas, **not** the write tool) and `call_tool(name, args) -> str` (args may be a dict or a JSON string). Paste your Part A functions where marked, and paste the schemas as the comment above `TOOLS` explains. Running the cell writes the file.

In [ ]:
%%writefile coapp/tools.py
"""Co. app v3 tools: read-only SQLite tools for the agent (from Lab 9 Part A)."""
import json
import re
import sqlite3

from itec632.corpus import db_path


class ToolError(Exception):
    pass


def rows(sql, params=()):
    con = sqlite3.connect(f"file:{db_path()}?mode=ro", uri=True)
    con.row_factory = sqlite3.Row
    try:
        return [dict(r) for r in con.execute(sql, params).fetchall()]
    finally:
        con.close()


CATEGORIES = ["Account/Access", "Email/M365", "Hardware", "Printing/Scanning", "Network/Internet", "Software",
              "Backup/Restore", "Onboarding/Offboarding", "Phone/VoIP", "Security Report",
              "Server/Infrastructure", "General Request"]
ASSET_TYPES = ["laptop", "desktop", "server", "firewall", "switch", "access_point", "printer", "phone", "nas"]
CLIENT_COLS = "code, name, industry, city, users, tier, client_since, sites, backup_included, has_voip"


# TODO: replace the five placeholder lines below with your Part A code: describe_minutes, check_client_code,
#       and the five tools lookup_ticket, search_tickets, get_client, list_assets, sla_target.
#       (rows, ToolError, CATEGORIES, ASSET_TYPES and CLIENT_COLS above are already the same as in Part A.)
def lookup_ticket(ticket_id): raise NotImplementedError
def search_tickets(client_code=None, status=None, category=None, since=None): raise NotImplementedError
def get_client(code): raise NotImplementedError
def list_assets(client_code, type=None, os_contains=None, include_retired=False): raise NotImplementedError
def sla_target(tier, priority): raise NotImplementedError


FUNCTIONS = {"lookup_ticket": lookup_ticket, "search_tickets": search_tickets, "get_client": get_client,
             "list_assets": list_assets, "sla_target": sla_target}

# TODO: the 5 read-only schemas. In a Part A cell, run
#   print(json.dumps([TOOL_SCHEMAS[n] for n in ["lookup_ticket", "search_tickets", "get_client", "list_assets", "sla_target"]]))
# and paste the printed text between the triple quotes. (json.loads turns it into Python; pasting the JSON
# directly would fail, because JSON writes false/true where Python writes False/True.)
TOOLS = json.loads(r"""
[]
""")


def call_tool(name, args):
    """Validate lightly, run, and ALWAYS return a string."""
    if name not in FUNCTIONS:
        return json.dumps({"error": f"Unknown tool {name!r}. Available: {list(FUNCTIONS)}"})
    if isinstance(args, str):
        try:
            args = json.loads(args or "{}")
        except json.JSONDecodeError:
            return json.dumps({"error": "Arguments were not valid JSON."})
    args = {k: v for k, v in (args or {}).items() if v is not None}
    schema = next((t["function"]["parameters"] for t in TOOLS if t["function"]["name"] == name), None)
    if schema is not None:
        from jsonschema import Draft202012Validator
        problems = [e.message for e in Draft202012Validator(schema).iter_errors(args)]
        if problems:
            return json.dumps({"error": f"Invalid arguments for {name}: " + "; ".join(problems)})
    try:
        return json.dumps(FUNCTIONS[name](**args), default=str)[:4000]
    except ToolError as e:
        return json.dumps({"error": str(e)})
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})

**B3 (TODO), part 1: update the prompt for tools.** The `prompts.py` you copied in B1 is Lab 8's, written for documents only: its first rule says to answer *only* from the documents in `<context>`, and another rule says "I don't know" when the documents don't have the answer. On the `data` route there are no documents, so a real model following those rules may refuse or ignore the tool results. Open `coapp/prompts.py` in the JupyterLab editor and:
1. change the first rule so tool results count as a valid source, alongside the documents;
2. add a rule telling the model to use the tools for specific clients, tickets, assets and SLA targets, and never to guess an ID, a client code, or a count;
3. bump `PROMPT_VERSION` and save.

(`coapp_reference/prompts.py` in this folder shows one way to word it.)

**B3 (TODO), part 2:** the new `coapp/assistant.py`. It uses only the shared spec from your v2 (`config.MODEL`, `prompts.build_messages`, `retrieve.build_index` / `retrieve`) plus your new tools. Finish `route(question)`, the **router** that picks a path for each question: return `"data"` when the question mentions a ticket ID, an asset tag, a known client, or data words (tickets, assets, servers...); `"both"` when it *also* has document words (how do I, policy, price...); otherwise `"docs"`. The regular expressions (text patterns, for example one that spots a ticket ID) are written for you.

In [ ]:
%%writefile coapp/assistant.py
"""Co. app v3: retrieval + tools with a keyword router (Lab 9 Part B)."""
import json
import re
import sqlite3
import time

from itec632 import NO_THINK, get_client as make_llm_client, strip_think   # not the get_client tool
from itec632.corpus import db_path, load_docs

from . import config
from .prompts import build_messages
from .retrieve import build_index, retrieve
from .tools import TOOLS, call_tool

TICKET_OR_ASSET = re.compile(r"\bTCK-\d{4,7}\b|\bLR-[A-Z]{3}-\d{4}\b", re.I)
DATA_WORDS = re.compile(r"\b(tickets?|assets?|laptops?|desktops?|servers?|firewalls?|switch(es)?|inventory|"
                        r"assigned|hostnames?|asset tags?|warranty|how many users)\b", re.I)
DOCS_WORDS = re.compile(r"\b(how (do|should|can) (i|we)|how to|procedure|policy|runbook|steps?|price|pricing|"
                        r"cost|rate|pto|holiday|business hours|handbook|escalat\w*|includes?)\b", re.I)
_con = sqlite3.connect(f"file:{db_path()}?mode=ro", uri=True)
CLIENTS = _con.execute("SELECT code, lower(name) FROM clients").fetchall()
_con.close()


def mentions_client(question):
    q = question.lower()
    return any(re.search(rf"\b{code}\b", question) or name in q or " ".join(name.split()[:2]) in q
               for code, name in CLIENTS)


def route(question):
    """'docs', 'data' or 'both'."""
    data = False   # TODO: True if TICKET_OR_ASSET or DATA_WORDS matches, or mentions_client(question)
    docs = False   # TODO: True if DOCS_WORDS matches
    # TODO: return the route name ("both", "data" or "docs") that matches the two flags
    return "docs"


_INDEX = None


def answer(question, history=None, *, model=None, timeout=None):
    global _INDEX
    t0 = time.perf_counter()
    model = model or config.MODEL
    llm = make_llm_client()
    which = route(question)
    chunks = []
    if which in ("docs", "both"):
        if _INDEX is None:
            _INDEX = build_index(load_docs())
        chunks = retrieve(_INDEX, question, k=5)
    messages = build_messages(question, chunks, history)
    tools = TOOLS if which in ("data", "both") else None
    tool_calls, text = [], ""
    for step in range(5):
        offer = tools is not None and step < 4
        resp = llm.chat.completions.create(model=model, messages=messages, temperature=0.2, max_tokens=700,
                                           extra_body=NO_THINK, **({"tools": tools} if offer else {}))
        msg = resp.choices[0].message
        if not (offer and msg.tool_calls):
            text = strip_think(msg.content or "")
            break
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments or "{}"}}
            for tc in msg.tool_calls]})
        for tc in msg.tool_calls:
            result = call_tool(tc.function.name, tc.function.arguments or "{}")
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            tool_calls.append({"name": tc.function.name, "args": args})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    seen, sources = set(), []
    for c in chunks:
        if (c["doc_id"], c["version"]) not in seen:
            seen.add((c["doc_id"], c["version"]))
            sources.append({"doc_id": c["doc_id"], "version": c["version"], "title": c["title"]})
    return {"answer": text, "sources": sources, "tool_calls": tool_calls, "model": model,
            "latency_s": round(time.perf_counter() - t0, 3), "route": which}

**B4.** Bonus checks. These reload your `coapp` package, call `answer()` on one data question and one document question, and try the router on a few more, printing the route each question would ideally take next to yours. A keyword router gets some of these wrong even when your code is correct: "server management cost" is a price question, but "server" is also a data word. That's a limit of keyword routing worth mentioning in your analysis, not a bug.

**Requests and time:** the first document question builds the retrieval index. This folder has its own `.coapp_cache/`, so the corpus is embedded again: about 26 batched requests and 1-2 minutes of waiting with no output. After that, the two questions take about 5 more requests, and reruns reuse the cache. Until `coapp/tools.py` has `TOOLS`, this cell skips the `answer()` calls, so your first Run All doesn't spend minutes building a document index for an unfinished bonus.

In [ ]:
def fresh_coapp():
    for name in [m for m in sys.modules if m == "coapp" or m.startswith("coapp.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    return importlib.import_module("coapp.tools"), importlib.import_module("coapp.assistant")

try:
    my_tools, my_assistant = fresh_coapp()
except Exception as e:
    print("Could not import coapp:", type(e).__name__, e)
    my_tools = my_assistant = None

names = {t["function"]["name"] for t in getattr(my_tools, "TOOLS", [])}
ok_call = bad_call = ""
if my_tools:
    ok_call = my_tools.call_tool("sla_target", {"tier": "Premier", "priority": "P1"})
    bad_call = my_tools.call_tool("lookup_ticket", '{"ticket_id": 42}')
    print(ok_call[:150], "\n", bad_call[:150])
ckb.check("B_tools_inventory", names == {"lookup_ticket", "search_tickets", "get_client", "list_assets", "sla_target"}, 2,
          "coapp/tools.py must define TOOLS with exactly the 5 read-only tools.")
ckb.check("B_call_tool", '"error"' not in ok_call and '"response_minutes":15' in ok_call.replace(" ", "") and '"error"' in bad_call, 2,
          "call_tool must return JSON results for valid calls and an error JSON for invalid arguments.")

KEYS = {"answer", "sources", "tool_calls", "model", "latency_s"}
data_out = doc_out = {}
if my_assistant and not names:
    print("Part B not started (coapp/tools.py has no TOOLS yet): skipping the answer() calls.")
if my_assistant and names and "tool" not in str(getattr(sys.modules.get("coapp.prompts"), "SYSTEM_PROMPT", "")).lower():
    print("Note: your SYSTEM_PROMPT never mentions tools. Do B3 part 1, or a real model may refuse data questions.")
if my_assistant and names:
    for label, qtext in [("data", "Which open tickets does Kaw Valley Auto Group have?"),
                         ("docs", "How many PTO days does a new employee get in their first two years?")]:
        try:
            out = my_assistant.answer(qtext)
        except Exception as e:
            print(f"answer() raised on the {label} question: {type(e).__name__}: {e}")
            out = {}
        print(f"[{out.get('route')}] {qtext}\n  tools={out.get('tool_calls')}\n  sources={[s.get('doc_id') for s in out.get('sources', [])]}\n  {str(out.get('answer'))[:200]}")
        data_out, doc_out = (out, doc_out) if label == "data" else (data_out, out)
        time.sleep(PACE_S)
    for qq, expected in [("Who is assigned to TCK-119956?", "data"), ("What does server management cost per month?", "docs"),
                         ("How do I clear a stuck print spooler?", "docs"), ("What SLA applies to Bluffline's open tickets?", "data")]:
        print(f"route: {my_assistant.route(qq):5} (ideal {expected:4}) <- {qq}")
ckb.check("B_answer_dict", KEYS <= set(data_out) and KEYS <= set(doc_out), 2,
          "answer() must return a dict with answer, sources, tool_calls, model, latency_s.")
ckb.check("B_data_question_uses_tools", len(data_out.get("tool_calls", [])) >= 1, 2,
          "A ticket question should route to tools (route() -> 'data') and make at least one tool call.")
ckb.check("B_doc_question_uses_retrieval", len(doc_out.get("sources", [])) >= 1 and len(doc_out.get("tool_calls", [1])) == 0, 2,
          "A PTO policy question should route to 'docs': sources from retrieval and no tool calls.")

## Written analysis (40 points)

Answer in the cells below. Use **your own numbers and traces** from this notebook, and cite the chapter (section names or concepts from Chapter 6, "Agents" and "Agent Failure Modes and Evaluation").

**Q1. Planning and tool-selection failures.** Pick two tasks from A6/A7 that failed or were inefficient (if none failed, use the two longest traces). For each: what the agent did step by step, what it should have done, and which failure category from the chapter it belongs to: a planning failure (either a tool use error such as an invalid tool, invalid parameters, or incorrect parameter values, or a goal failure), a tool failure (the right tool returned a wrong output, or a needed tool was missing), or an efficiency problem. Did the error messages from A3 help it recover? Point to the trace.

*Your answer:*

**Q2. Efficiency numbers.** Report your mean steps, mean extra steps, and mean tokens per task, and compare single-step vs multi-step tasks. Why do tokens grow faster than the number of steps in an agent loop? (Look at what `run_agent` re-sends each step.) Propose one concrete change to the tools or prompt that would cut steps or tokens for a specific task, and name the tradeoff, connecting it to the chapter's discussion of tool inventories and tool selection.

*Your answer:*

**Q3. Model comparison.** Using your A8 table, compare qwen-moe, gemma-moe and qwen-small on success, tool errors, steps and latency. Where did the small model break down (planning or tool use)? Based on your evidence, would you let `qwen-small` handle single-lookup questions in production to save cost? What would you need to measure first?

*Your answer:*

**Q4. Evaluating agents, and write actions.** Your scorer checks that the final answer contains the right facts. Name two things this misses that the chapter says to evaluate for agents (for example, plan validity, tool-call validity rates, or cost/latency per task), and show how you could compute one of them from the `traces` you already have. Then: in A5, what did the agent do when the human declined? Name one Larkspur action (beyond closing tickets) that should always require confirmation and why, citing the chapter's point about write actions.

*Your answer:*

## AI-use reflection (10 points)

1. What did you use AI assistance for in this lab (tools, which steps)?

*Your answer:*

2. Where did the AI help most, and where was it wrong or unhelpful? How did you notice?

*Your answer:*

3. What is one thing you now understand that you could explain without AI help?

*Your answer:*

4. Copilot Chat is itself an agent that plans and calls tools. Describe one moment (in this lab or before) where an AI coding assistant showed one of this chapter's failure modes (a planning failure, a wrong tool or argument, or an inefficient loop). What would have caught it?

*Your answer:*

## Submit

Run this last, save, upload the .ipynb. (Part B bonus points are listed in the summary as `partB_bonus_points`.)

In [ ]:
got_b, max_b = ckb.score()
ck.record("partB_bonus_points", got_b)
ck.record("partB_bonus_max", max_b)
ck.report()